In [ ]:
import pandas as pd

In [ ]:
!pip install stop-words

In [ ]:
!pip install pymorphy3

In [ ]:
df = pd.read_csv(
    "all_resumes_adv-cl.csv",
    encoding="utf-8",
    engine="python",
    on_bad_lines="skip",
)

In [ ]:
df = df.dropna(subset=['text', 'label']).copy()
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(str)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['label'] = le.fit_transform(df['label'])

In [ ]:
df.head()

,label,text
0,2,личные качества порядочность пунктуальность fr...
1,0,я студент хочу работать сфере back end разрабо...
2,2,a1 игорь frontend разработчик контакты temonav...
3,3,c2 11 years experience in project management e...
4,1,c1 люблю решать сложные задачи используя прост...


In [ ]:
print("Пример текста:\n", df["text"].head(2).tolist())

Пример текста:
 ['личные качества порядочность пунктуальность frontend разработчик высшее образование кпу запорожье experience years 12 33', 'я студент хочу работать сфере back end разработки или data science хорошо взаимодействую людьми постоянно самообучаюсь ответственный стратегическое мышление играю шахматы умение работать режиме многозадачности высокие аналитические способности позволяют мне эффективно работать большими объёмами информации быстро находить качественные решения сложных задач back end разработчик неоконченное высшее образование национальный исследовательский технологический университет мисис москва москва']


In [ ]:
import pandas as pd
from string import punctuation
from stop_words import get_stop_words
from pymorphy3 import MorphAnalyzer
import re
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.15,
    stratify=df['label'],
    random_state=42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp,
    test_size=0.176,  # 0.176 * 0.85 ≈ 0.15
    stratify=y_temp,
    random_state=42
  )

In [ ]:
text_corpus_train = X_train.values
text_corpus_valid = X_valid.values
text_corpus_test = X_test.values

In [ ]:
print(text_corpus_train)

['c2 являюсь python разработчиком начального уровня умею писать красивый понятный код также умею искать информацию для решения поставленных задач во время работы над проектами познакомился фреймворком flask виртуальным окружением различными методиками тестирования контроля качества исходного кода с веб сервисом для хостинга it проектов github рассматриваю должность стажера или junior backend разработчика готов упорно усердно учиться ссылка на мой github аккаунт python backend разработчик неоконченное высшее образование российский университет транспорта москва москва'
 'c1 обучение на заочном отделении нужен учебный отпуск на летнюю сессию уверенно знаю js node js работал c cpp delphy python неплохо разбираюсь устройством браузера особенно по части как формировать запросы чтобы потом их делать автоматически основном занимался интернет разработками парсинг анализ данных агрегатор данных автоматизация человеческих действий интернете так же неплохо разбираюсь программном устройстве компьют

In [ ]:
import numpy as np
import keras
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Input, Embedding, Conv1D, GlobalMaxPool1D, SimpleRNN, LSTM, GRU, Masking
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import TensorBoard
from tensorflow.keras.losses import categorical_crossentropy
from keras.callbacks import EarlyStopping

In [ ]:
tokenizer = Tokenizer(num_words=None,
                     filters='#$%&()*+-<=>@[\\]^_`{|}~\t\n',
                     lower = False, split = ' ')
tokenizer.fit_on_texts(text_corpus_train)

#reform all texts to index sequances
sequences_train = tokenizer.texts_to_sequences(text_corpus_train)
sequences_val = tokenizer.texts_to_sequences(text_corpus_valid)
sequences_test = tokenizer.texts_to_sequences(text_corpus_test)

word_count = len(tokenizer.index_word) + 1 #Count of words in dict
print(word_count)
training_length = max([len(i.split()) for i in text_corpus_train]) #max text length
print(training_length)

24933
782


In [ ]:
X_train = pad_sequences(sequences_train, maxlen=training_length)
X_valid = pad_sequences(sequences_val, maxlen=training_length)
X_test = pad_sequences(sequences_test, maxlen=training_length)

**1**

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(

In [ ]:
num_classes = len(le.classes_)

model = Sequential()
model.add(
    Embedding(
        input_dim=word_count,
        output_dim=30,
        mask_zero=True,
        trainable=True
    )
)

model.add(SimpleRNN(64))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
from keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=10,
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step - accuracy: 0.2676 - loss: 1.3907 - val_accuracy: 0.3746 - val_loss: 1.3335
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.4064 - loss: 1.3012 - val_accuracy: 0.3762 - val_loss: 1.2912
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.4391 - loss: 1.2101 - val_accuracy: 0.3925 - val_loss: 1.2352
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.5478 - loss: 1.0751 - val_accuracy: 0.5033 - val_loss: 1.1346
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.6727 - loss: 0.9045 - val_accuracy: 0.5619 - val_loss: 1.0446
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.8059 - loss: 0.6933 - val_accuracy: 0.6417 - val_loss: 0.8789
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.8872 - loss: 0.4893 - val_accuracy: 0.6954 - val_loss: 0.7591
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9266 - loss: 0.3254 - val_accuracy: 0.7182 - val_loss: 0.7385
Epoch 9/10

In [ ]:
print("len(X_test):", len(X_test))
print("len(y_test):", len(y_test))

len(X_test): 615
len(y_test): 615


In [ ]:
score = model.evaluate(X_test, y_test, batch_size=512, verbose=1)

print("\nTest loss:", round(score[0], 4))
print("Test accuracy:", round(score[1] * 100, 2), "%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.7938 - loss: 0.5911

Test loss: 0.5883
Test accuracy: 79.51 %


**2**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

num_classes = 4

model = Sequential()
model.add(
    Embedding(
        input_dim=word_count,
        output_dim=30,
        mask_zero=True,
        trainable=True
    )
)
model.add(LSTM(64, recurrent_dropout=0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=10,
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 38s 5s/step - accuracy: 0.2966 - loss: 1.3827 - val_accuracy: 0.3762 - val_loss: 1.3672
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 38s 5s/step - accuracy: 0.3839 - loss: 1.3581 - val_accuracy: 0.3762 - val_loss: 1.3222
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step - accuracy: 0.3791 - loss: 1.3065 - val_accuracy: 0.3762 - val_loss: 1.2433
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 45s 5s/step - accuracy: 0.4069 - loss: 1.2225 - val_accuracy: 0.4593 - val_loss: 1.0962
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 39s 5s/step - accuracy: 0.5217 - loss: 1.0519 - val_accuracy: 0.6401 - val_loss: 0.9260
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 29s 5s/step - accuracy: 0.6279 - loss: 0.9050 - val_accuracy: 0.6629 - val_loss: 0.8564
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 40s 5s/step - accuracy: 0.6558 - loss: 0.8101 - val_accuracy: 0.6401 - val_loss: 0.7728
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 42s 5s/step - accuracy: 0.6543 - loss: 0.7296 - val_accuracy: 0.6547 - val_loss: 0.6829
Epoch 9/

In [ ]:
score = model.evaluate(X_test, y_test, batch_size=512, verbose=1)

print("\nTest loss:", round(score[0], 4))
print("Test accuracy:", round(score[1] * 100, 2), "%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 285ms/step - accuracy: 0.8005 - loss: 0.5278

Test loss: 0.5193
Test accuracy: 80.33 %


***3***

In [ ]:
model = Sequential()

model.add(
    Embedding(
        input_dim=word_count,
        output_dim=30,
        trainable=True,
        mask_zero=True
    )
)
model.add(GRU(64, recurrent_dropout=0.2))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=10,
    verbose=1,
    callbacks=[early_stopping]
)

Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 4s/step - accuracy: 0.3445 - loss: 1.3755 - val_accuracy: 0.3762 - val_loss: 1.3617
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 4s/step - accuracy: 0.3678 - loss: 1.3556 - val_accuracy: 0.3762 - val_loss: 1.3358
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 4s/step - accuracy: 0.3774 - loss: 1.3316 - val_accuracy: 0.3762 - val_loss: 1.3116
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.3736 - loss: 1.3113 - val_accuracy: 0.3762 - val_loss: 1.2993
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 4s/step - accuracy: 0.3675 - loss: 1.3037 - val_accuracy: 0.3762 - val_loss: 1.2747
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.3657 - loss: 1.2627 - val_accuracy: 0.3811 - val_loss: 1.1923
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 24s 4s/step - accuracy: 0.4157 - loss: 1.1489 - val_accuracy: 0.5147 - val_loss: 1.0187
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 0.5416 - loss: 0.9987 - val_accuracy: 0.5033 - val_loss: 0.9493
Epoch 9/

In [ ]:
score = model.evaluate(X_test, y_test, batch_size=512, verbose=1)

print("\nTest loss:", round(score[0], 4))
print("Test accuracy:", round(score[1] * 100, 2), "%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 415ms/step - accuracy: 0.6194 - loss: 0.8523

Test loss: 0.8421
Test accuracy: 62.44 %
